# 05b — OntoBricks GraphRAG MCP Server

This notebook deploys a **graph-backed Retrieval-Augmented Generation (GraphRAG) service** as a Databricks App, then exposes it to the Multi-Agent Supervisor via an MCP (Model Context Protocol) connection. Unlike the Neo4j approach in notebook `05b_create_mcp_neo4j_graph_OPTIONAL`, everything here runs **entirely within the Databricks environment** — no external graph database required.

---

## Architecture

```
┌─────────────────────────────────┐
│  Supervisor Agent (notebook 04) │
│  routes queries to child agents │
└───────────┬─────────────────────┘
            │  MCP protocol (SSE)
            ▼
┌───────────────────────────────────────┐
│  UC Connection: ontobricks-graphrag   │
│  (HTTP + bearer token auth)           │
└───────────┬───────────────────────────┘
            │
            ▼
┌───────────────────────────────────────┐
│  Databricks App (FastMCP server)      │
│  Tools:                               │
│    • query_graph(query)               │
│    • get_company_summary(ticker)      │
│    • compare_companies(tickers)       │
└───────────┬───────────────────────────┘
            │  Spark SQL via
            │  databricks-connect
            ▼
┌───────────────────────────────────────┐
│  Unity Catalog Delta Tables           │
│    • graphrag_vertices (1,757 rows)   │
│    • graphrag_edges    (3,493 rows)   │
└───────────────────────────────────────┘
```

### Why GraphRAG instead of plain SQL?

Graph structures let the Supervisor Agent answer **relationship-centric** questions that are awkward in flat SQL:
* "Which companies traded above their 5-day moving average for the longest streak?" → NEXT_DAY chain traversal
* "Compare the fundamentals of NVDA vs AAPL" → Company node properties
* "Show me TSLA's most volatile trading days" → TradingDay properties via TRADED_ON

The graph schema encodes **temporal chains** (NEXT_DAY) and **ownership links** (TRADED_ON) that make multi-hop reasoning natural.

---

## Graph Schema — Nodes

### Company (7 nodes)
One per Mag-7 stock. Carries the latest fundamental snapshot.

| Property | Type | Example | Description |
|----------|------|---------|-------------|
| `id` / `ticker` | string | `NVDA` | Stock ticker (also the node ID) |
| `market_cap` | long | 3,593,883,705,344 | Market capitalization in USD |
| `beta` | double | 1.637 | Volatility relative to S&P 500 |
| `pe_trailing` | double | 54.52 | Trailing 12-month P/E ratio |
| `pe_forward` | double | 31.25 | Forward P/E ratio |
| `ev_ebitda` | double | 48.27 | Enterprise Value / EBITDA |

### TradingDay (1,750 nodes)
One per (company × trading date). \~250 trading days per company.

| Property | Type | Example | Description |
|----------|------|---------|-------------|
| `id` | string | `NVDA_2024-11-25` | Composite key: `{ticker}_{date}` |
| `ticker` | string | `NVDA` | Owning company |
| `date` | timestamp | 2024-11-25 | Calendar date |
| `price_close` | double | 141.95 | Closing price in USD |
| `volume` | int | 198,766,300 | Shares traded |
| `daily_return` | double | 0.83 | Intraday return % = (close-open)/open×100 |

---

## Graph Schema — Edges

| Relationship | Source → Target | Count | Purpose |
|-------------|-----------------|-------|---------|
| **TRADED_ON** | Company → TradingDay | 1,750 | Links a company to each of its trading days |
| **NEXT_DAY** | TradingDay → TradingDay | 1,743 | Temporal chain within each company (7 fewer = 7 chain starts) |

The **NEXT_DAY** chain enables streak/momentum queries: traverse consecutive days to find winning streaks, drawdowns, or volatility clusters without complex window functions.

---

## MCP Tools Exposed

The Databricks App runs a [FastMCP](https://github.com/jlowin/fastmcp) server over SSE transport with three tools:

| Tool | Signature | Use Case |
|------|-----------|----------|
| `query_graph` | `query_graph(query: str)` | Natural-language graph queries — routes to company lookups, comparisons, or schema info based on keywords |
| `get_company_summary` | `get_company_summary(ticker: str)` | Deep-dive on one company: fundamentals + last 10 trading days |
| `compare_companies` | `compare_companies(tickers: str)` | Side-by-side comparison table; pass comma-separated tickers like `"AAPL,MSFT,NVDA"` |

---

## Cell Execution Order

| Cell | Action | Idempotent? |
|------|--------|-------------|
| 2 | Install `fastmcp` (for local testing only) | Yes |
| 3 | Load config, set names, init SDK client | Yes |
| 4 | Build graph DataFrames from `ticker_data` | Yes |
| 5 | Write `graphrag_vertices` and `graphrag_edges` Delta tables | Yes (overwrite) |
| 6 | Generate MCP app source code string | Yes |
| 7 | Upload files → create app → deploy | Yes (delete-before-upload pattern) |
| 8 | Create UC HTTP connection with bearer token | Skips if exists |
| 9 | Smoke-test the graph tables directly | Yes |

---

## Prerequisites

1. Run **`00_tables_setup`** — creates the `ticker_data` source table in Unity Catalog
2. **Databricks Apps** must be enabled in the workspace (contact admin if `POST /api/2.0/apps` returns 404)
3. **`../config.py`** must define: `catalog`, `schema`, `app_resource_suffix`
4. The executing user needs `CREATE TABLE` on the target schema and `CREATE CONNECTION` on the metastore

## Troubleshooting

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| App stays `UNAVAILABLE` for >5 min | Workspace Apps quota or cold start | Check Apps UI; re-run cell 7 |
| `CONNECTION_HTTP_BEARER` error | Missing `bearer_token` in connection options | Cell 8 includes the token automatically |
| MCP tools return empty results | `graphrag_vertices` table missing or empty | Re-run cells 4–5 |
| Supervisor can't reach MCP | Connection name mismatch | Verify `ontobricks-graphrag` in cell 8 output matches notebook 04 config |

In [0]:
# --- Dependencies ---
# fastmcp is only needed for local testing/validation of the MCP server. The Databricks App installs its
# own copy from requirements.txt (see cell 6). The OntoBricks/ontos package is NOT installed here because
# it runs inside the Databricks App runtime, not in this notebook's compute context.
%pip install fastmcp --quiet
dbutils.library.restartPython()

In [0]:
# --- Imports & Configuration ---
# Standard libraries plus PySpark for graph DataFrame construction (cells 4-5) and the Databricks SDK for
# REST API calls to manage Apps and Connections (cells 7-8).
import sys
import os
import json
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from databricks.sdk import WorkspaceClient

# Load shared project config (catalog, schema, app_resource_suffix, sa_name). These variables are defined
# in ../config.py and shared across all setup notebooks.
sys.path.append('..')
exec(open('../config.py').read())

CATALOG = catalog
SCHEMA = schema

# Derive app and connection names from the shared suffix so they're unique per workshop participant but
# deterministic for re-runs and cross-notebook references. Both names include the suffix to prevent
# collisions when multiple users share a workspace. Notebook 04 discovers connections by scanning UC, so
# the suffixed name is picked up automatically.
GRAPHRAG_APP_NAME = f"ontobricks-graphrag-{app_resource_suffix}"
MCP_CONNECTION_NAME = f"ontobricks-graphrag-{app_resource_suffix}"

# WorkspaceClient auto-authenticates using the notebook's execution context. We use w.api_client.do() for
# REST calls throughout because the SDK's high-level methods (w.apps.*, w.connections.*) are incomplete on
# serverless compute.
w = WorkspaceClient()
print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")
print(f"App:     {GRAPHRAG_APP_NAME}")
print(f"MCP:     {MCP_CONNECTION_NAME}")

In [0]:
# --- Build Graph DataFrames ---
# Transform the flat ticker_data table (created by 00_tables_setup) into a property-graph representation
# with two vertex types and two edge types. This mirrors the Neo4j schema from notebook
# 05b_create_mcp_neo4j_graph but stores everything as Delta tables the Databricks App queries via Spark SQL.

# ── Source Data ──
# ticker_data has one row per (company × trading day) with columns: Date, company_name, price_open,
# price_close, volume, pe_trailing, pe_forward, peg, ev_ebitda, market_cap, beta
ticker_df = spark.table(f"{CATALOG}.{SCHEMA}.ticker_data")
print(f"Ticker data: {ticker_df.count()} rows")

# Get the universe of companies (should be 7 Mag-7 stocks)
companies = [row.company_name for row in ticker_df.select("company_name").distinct().collect()]
print(f"Companies: {companies}")

# ── Company Vertices ──
# One node per company with static fundamental metrics. These values are the latest snapshot from Yahoo
# Finance (loaded by 00_tables_setup). We use company_name as the node ID for simple lookups.
companies_df = ticker_df.select(
    F.col("company_name").alias("id"),
    F.lit("Company").alias("type"),
    F.col("company_name").alias("ticker"),
    "market_cap", "beta", "pe_trailing", "pe_forward", "ev_ebitda"
).dropDuplicates(["id"])

# ── TradingDay Vertices ──
# One node per (company × date) with intraday price data. The composite ID format "TICKER_YYYY-MM-DD"
# enables efficient filtering and ensures uniqueness across companies. daily_return = intraday %.
trading_days_df = ticker_df.select(
    F.concat_ws("_", "company_name", F.date_format("Date", "yyyy-MM-dd")).alias("id"),
    F.lit("TradingDay").alias("type"),
    F.col("company_name").alias("ticker"),
    F.col("Date").alias("date"),
    "price_open", "price_close", "volume",
    F.round((F.col("price_close") - F.col("price_open")) / F.col("price_open") * 100, 4).alias("daily_return")
)

# ── TRADED_ON Edges ──
# Links each Company to its TradingDay nodes. Primary ownership relationship enabling "show me NVDA's
# trading history" queries.
traded_on_edges = ticker_df.select(
    F.col("company_name").alias("src"),
    F.concat_ws("_", "company_name", F.date_format("Date", "yyyy-MM-dd")).alias("dst"),
    F.lit("TRADED_ON").alias("relationship")
)

# ── NEXT_DAY Edges ──
# Temporal chain: each TradingDay points to the next calendar trading day for the SAME company. This enables
# streak/momentum queries like "longest consecutive days of positive returns" via graph traversal instead of
# window functions. We use lead() over a per-company date-ordered window to find the next date, then filter
# out the last day per company (where next_date is null).
day_window = Window.partitionBy("company_name").orderBy("Date")
with_next = ticker_df.withColumn("next_date", F.lead("Date").over(day_window))
next_day_edges = with_next.filter(F.col("next_date").isNotNull()).select(
    F.concat_ws("_", "company_name", F.date_format("Date", "yyyy-MM-dd")).alias("src"),
    F.concat_ws("_", "company_name", F.date_format("next_date", "yyyy-MM-dd")).alias("dst"),
    F.lit("NEXT_DAY").alias("relationship")
)

# Expect: 7 Company vertices, ~1750 TradingDay vertices, ~1750 TRADED_ON edges, ~1743 NEXT_DAY edges
# (7 fewer = one chain-end per company)
print(f"\nGraph components:")
print(f"  Company vertices: {companies_df.count()}")
print(f"  TradingDay vertices: {trading_days_df.count()}")
print(f"  TRADED_ON edges: {traded_on_edges.count()}")
print(f"  NEXT_DAY edges: {next_day_edges.count()}")

In [0]:
# --- Persist Graph as Delta Tables ---
# We flatten all vertex types into a single table with a union-compatible schema. Properties that don't
# apply to a given vertex type are stored as NULL. All property columns are cast to STRING for uniform
# storage; the MCP server casts them back when building query results (see cell 6).
#
# This "wide + sparse" layout is simpler than separate tables per vertex type and works well because the
# Databricks App issues targeted SQL with WHERE type = '...'.

vertices_table = f"{CATALOG}.{SCHEMA}.graphrag_vertices"
edges_table = f"{CATALOG}.{SCHEMA}.graphrag_edges"

# ── Vertices: union Company + TradingDay nodes ──
# Company nodes carry fundamentals (market_cap, beta, pe_*) but no date/price props. TradingDay nodes carry
# date/price/volume/return but no fundamental props.
all_vertices = companies_df.select("id", "type", "ticker", 
    F.col("market_cap").cast("string").alias("prop_market_cap"),
    F.col("beta").cast("string").alias("prop_beta"),
    F.col("pe_trailing").cast("string").alias("prop_pe_trailing"),
    F.col("pe_forward").cast("string").alias("prop_pe_forward"),
    F.col("ev_ebitda").cast("string").alias("prop_ev_ebitda"),
    F.lit(None).cast("string").alias("prop_date"),
    F.lit(None).cast("string").alias("prop_price_close"),
    F.lit(None).cast("string").alias("prop_volume"),
    F.lit(None).cast("string").alias("prop_daily_return")
).unionByName(
    trading_days_df.select("id", "type", "ticker",
        F.lit(None).cast("string").alias("prop_market_cap"),
        F.lit(None).cast("string").alias("prop_beta"),
        F.lit(None).cast("string").alias("prop_pe_trailing"),
        F.lit(None).cast("string").alias("prop_pe_forward"),
        F.lit(None).cast("string").alias("prop_ev_ebitda"),
        F.col("date").cast("string").alias("prop_date"),
        F.col("price_close").cast("string").alias("prop_price_close"),
        F.col("volume").cast("string").alias("prop_volume"),
        F.col("daily_return").cast("string").alias("prop_daily_return")
    )
)

# ── Edges: union TRADED_ON + NEXT_DAY relationships ──
# Both edge types share the same (src, dst, relationship) schema.
all_edges = traded_on_edges.unionByName(next_day_edges)

# Write with overwrite mode so the notebook is fully idempotent. Delta gives us ACID guarantees + time-travel.
all_vertices.write.format("delta").mode("overwrite").saveAsTable(vertices_table)
all_edges.write.format("delta").mode("overwrite").saveAsTable(edges_table)

print(f"✅ Wrote {vertices_table} ({all_vertices.count()} rows)")
print(f"✅ Wrote {edges_table} ({all_edges.count()} rows)")

In [0]:
# --- Generate MCP Server Application Code ---
# This cell builds the Python source for the Databricks App as a multi-line string. The app is a FastMCP
# server that exposes three tools over Server-Sent Events (SSE).
#
# Key design decisions:
#   1. The app uses databricks-connect (not direct JDBC) to query Delta tables via Spark SQL. This inherits
#      the app's service principal permissions and avoids managing credentials.
#   2. CATALOG/SCHEMA come from environment variables set in app.yaml (cell 7), making the app portable.
#   3. The query_graph tool uses keyword matching (not an LLM) to route queries. This keeps the app stateless
#      and fast. For more sophisticated NL→SQL, see the graphframes-graphrag notebook.
#   4. All string literals inside the app source use escaped quotes (\') because the entire app is defined
#      as a Python string in this notebook cell.
#   5. Fallback defaults for CATALOG/SCHEMA use __PLACEHOLDER__ sentinels replaced by .replace() at the end,
#      so the app source is fully parameterized to config.py without needing an f-string (which would break
#      the many {CATALOG}/{SCHEMA} runtime references inside the app code).

app_source = '''
import os
from fastmcp import FastMCP
from databricks.sdk import WorkspaceClient
from databricks.connect import DatabricksSession

# Initialize MCP server and Spark session. FastMCP handles SSE transport, tool registration, and JSON-RPC
# serialization. DatabricksSession connects back to the workspace's Spark cluster.
mcp = FastMCP("OntoBricks GraphRAG")
w = WorkspaceClient()
spark = DatabricksSession.builder.getOrCreate()

# Read table coordinates from environment (set in app.yaml by cell 7). Fallback defaults are injected from
# config.py at notebook build time via .replace() — never hardcoded.
CATALOG = os.environ.get("CATALOG", "__CATALOG_DEFAULT__")
SCHEMA = os.environ.get("SCHEMA", "__SCHEMA_DEFAULT__")
VERTICES_TABLE = f"{CATALOG}.{SCHEMA}.graphrag_vertices"
EDGES_TABLE = f"{CATALOG}.{SCHEMA}.graphrag_edges"

@mcp.tool()
def query_graph(query: str) -> str:
    """Query the stock knowledge graph using natural language.
    The graph contains Company nodes (AAPL, MSFT, NVDA, GOOGL, META, AMZN, TSLA)
    with fundamentals (market_cap, beta, pe_trailing, pe_forward, ev_ebitda),
    TradingDay nodes with prices/volumes/returns, and TRADED_ON/NEXT_DAY relationships.
    """
    # Route based on keywords — intentionally simple. The Supervisor Agent handles complex NL understanding;
    # this tool just needs structured dispatch.
    query_lower = query.lower()
    
    if any(word in query_lower for word in ["compare", "comparison", "vs", "versus"]):
        # Comparison mode: return all companies ranked by market cap
        result = spark.sql(f"""
            SELECT ticker, prop_market_cap as market_cap, prop_beta as beta,
                   prop_pe_trailing as pe_trailing, prop_pe_forward as pe_forward
            FROM {VERTICES_TABLE}
            WHERE type = \'Company\'
            ORDER BY CAST(prop_market_cap AS BIGINT) DESC
        """).toPandas()
        return f"Company Comparison:\\n{result.to_string(index=False)}"
    
    # Check for a specific ticker mention
    tickers = ["AAPL", "MSFT", "NVDA", "GOOGL", "META", "AMZN", "TSLA"]
    found_ticker = next((t for t in tickers if t.lower() in query_lower), None)
    
    if found_ticker:
        # Single-company mode: fundamentals + recent trading activity
        company = spark.sql(f"""
            SELECT * FROM {VERTICES_TABLE}
            WHERE type = \'Company\' AND ticker = \'{found_ticker}\'
        """).toPandas()
        
        recent = spark.sql(f"""
            SELECT prop_date as date, prop_price_close as close, 
                   prop_volume as volume, prop_daily_return as return_pct
            FROM {VERTICES_TABLE}
            WHERE type = \'TradingDay\' AND ticker = \'{found_ticker}\'
            ORDER BY prop_date DESC
            LIMIT 5
        """).toPandas()
        
        return f"=== {found_ticker} Summary ===\\n{company.to_string(index=False)}\\n\\nRecent Trading:\\n{recent.to_string(index=False)}"
    
    # Fallback: describe the graph schema so the agent can refine its query
    return f"""Graph contains:
- 7 Company nodes: AAPL, MSFT, NVDA, GOOGL, META, AMZN, TSLA
- ~1750 TradingDay nodes with prices and volumes
- TRADED_ON edges linking companies to trading days
- NEXT_DAY edges forming temporal chains

Ask about specific companies or request comparisons."""

@mcp.tool()
def get_company_summary(ticker: str) -> str:
    """Get a detailed summary of a company including fundamentals and recent trading data.
    Available tickers: AAPL, MSFT, NVDA, GOOGL, META, AMZN, TSLA
    """
    ticker = ticker.upper()
    
    # Fetch fundamental properties from the Company vertex
    company = spark.sql(f"""
        SELECT ticker, prop_market_cap as market_cap, prop_beta as beta,
               prop_pe_trailing as pe_trailing, prop_pe_forward as pe_forward,
               prop_ev_ebitda as ev_ebitda
        FROM {VERTICES_TABLE}
        WHERE type = \'Company\' AND ticker = \'{ticker}\'
    """).toPandas()
    
    if company.empty:
        return f"Company {ticker} not found. Available: AAPL, MSFT, NVDA, GOOGL, META, AMZN, TSLA"
    
    # Fetch the 10 most recent TradingDay nodes (via TRADED_ON relationship)
    recent = spark.sql(f"""
        SELECT prop_date as date, prop_price_close as close, 
               prop_volume as volume, prop_daily_return as return_pct
        FROM {VERTICES_TABLE}
        WHERE type = \'TradingDay\' AND ticker = \'{ticker}\'
        ORDER BY prop_date DESC
        LIMIT 10
    """).toPandas()
    
    # Format as a human-readable summary for the Supervisor Agent
    c = company.iloc[0]
    result = f"""=== {ticker} Summary ===
Market Cap: ${int(float(c[\'market_cap\'])):,}
Beta: {c[\'beta\']}
P/E Trailing: {c[\'pe_trailing\']}
P/E Forward: {c[\'pe_forward\']}
EV/EBITDA: {c[\'ev_ebitda\']}

Recent Trading (last 10 days):
{recent.to_string(index=False)}"""
    return result

@mcp.tool()
def compare_companies(tickers: str) -> str:
    """Compare multiple companies side by side.
    Pass comma-separated tickers, e.g., \'AAPL,MSFT,NVDA\'
    """
    # Parse and normalize the comma-separated ticker input
    ticker_list = [t.strip().upper() for t in tickers.split(",")]
    in_clause = ",".join(f"\'{t}\'" for t in ticker_list)
    
    # Pull fundamental properties for the requested subset, ranked by market cap
    result = spark.sql(f"""
        SELECT ticker, prop_market_cap as market_cap, prop_beta as beta,
               prop_pe_trailing as pe_trailing, prop_pe_forward as pe_forward,
               prop_ev_ebitda as ev_ebitda
        FROM {VERTICES_TABLE}
        WHERE type = \'Company\' AND ticker IN ({in_clause})
        ORDER BY CAST(prop_market_cap AS BIGINT) DESC
    """).toPandas()
    
    return f"=== Company Comparison ===\\n{result.to_string(index=False)}"

# Entry point: run the FastMCP server with SSE transport. The Databricks App runtime calls this when the
# container starts. SSE is the transport protocol the UC MCP connection uses to communicate with this server.
if __name__ == "__main__":
    mcp.run(transport="sse")
'''.replace("__CATALOG_DEFAULT__", CATALOG).replace("__SCHEMA_DEFAULT__", SCHEMA)

print("MCP Server source code generated")
print(f"Tools: query_graph, get_company_summary, compare_companies")
print(f"App defaults: CATALOG={CATALOG}, SCHEMA={SCHEMA}")

In [0]:
# --- Deploy the Databricks App ---
# This cell performs three operations: 1) Upload app source files to a workspace directory, 2) Create the
# Databricks App (or skip if it already exists), 3) Deploy the latest code to the running app.
#
# NOTE: We use w.api_client.do() for all REST calls because the SDK's high-level methods
# (w.apps.create_and_wait, w.workspace.upload for non-.py files) are incomplete or broken on serverless
# compute. See the Troubleshooting section in the overview cell for common issues.

import base64

# ── Resolve the workspace directory for app source files ──
current_user = w.current_user.me()
username = current_user.user_name
app_path = f"/Workspace/Users/{username}/apps/{GRAPHRAG_APP_NAME}"

w.api_client.do("POST", "/api/2.0/workspace/mkdirs", body={"path": app_path})  # ensure target dir exists

# ── Upload app source files ──
# Three files required by the Databricks App runtime: main.py (FastMCP server code from cell 6), app.yaml
# (tells the runtime how to start the app and injects env vars), and requirements.txt (Python packages
# installed before main.py runs).
#
# We use the workspace/import REST API with base64-encoded content and format=AUTO. The delete-before-upload
# pattern avoids type-mismatch errors when a previous run created the file as a different workspace object
# type (NOTEBOOK vs FILE).
files_to_upload = [
    ("main.py", app_source),
    ("app.yaml", f'''command:\n  - python\n  - main.py\n\nenv:\n  - name: CATALOG\n    value: "{CATALOG}"\n  - name: SCHEMA\n    value: "{SCHEMA}"\n'''),
    ("requirements.txt", "fastmcp\ndatabricks-sdk\ndatabricks-connect\npandas\n")
]

for filename, content in files_to_upload:
    file_path = f"{app_path}/{filename}"
    try:
        w.workspace.delete(file_path)  # delete existing to avoid NOTEBOOK-vs-FILE type mismatch
    except:
        pass
    w.api_client.do("POST", "/api/2.0/workspace/import", body={
        "path": file_path,
        "content": base64.b64encode(content.encode()).decode(),
        "format": "AUTO",
        "overwrite": True
    })
    print(f"✅ Uploaded {filename}")

print(f"\n✅ App files written to {app_path}")

# ── Create or retrieve the Databricks App ──
# Apps are workspace-scoped singletons identified by name. If already exists, skip straight to deployment.
try:
    app = w.api_client.do("GET", f"/api/2.0/apps/{GRAPHRAG_APP_NAME}")
    print(f"App '{GRAPHRAG_APP_NAME}' already exists")
except:
    print(f"Creating app '{GRAPHRAG_APP_NAME}'...")
    app = w.api_client.do("POST", "/api/2.0/apps", body={
        "name": GRAPHRAG_APP_NAME,
        "description": "OntoBricks GraphRAG MCP Server for stock knowledge graph queries"
    })
    # Poll until the app transitions out of UNAVAILABLE. First-time creation takes 2-5 min (container build).
    for _ in range(30):
        status = w.api_client.do("GET", f"/api/2.0/apps/{GRAPHRAG_APP_NAME}")
        state = status.get("status", {}).get("state", status.get("app_status", {}).get("state", "UNKNOWN"))
        if state in ("IDLE", "RUNNING", "ACTIVE", "DEPLOYED"):
            break
        print(f"  App state: {state}")
        time.sleep(10)
    print(f"✅ App created")

# ── Deploy the latest source code ──
# Each deploy creates an immutable snapshot. Runtime pulls source, installs requirements.txt, runs main.py.
print(f"Deploying app...")
deployment = w.api_client.do("POST", f"/api/2.0/apps/{GRAPHRAG_APP_NAME}/deployments", body={
    "source_code_path": app_path
})
deployment_id = deployment.get("deployment_id", "")

for _ in range(60):  # poll deployment status until terminal state
    dep = w.api_client.do("GET", f"/api/2.0/apps/{GRAPHRAG_APP_NAME}/deployments/{deployment_id}")
    state = dep.get("status", {}).get("state", dep.get("deployment_status", {}).get("state", "UNKNOWN"))
    if state in ("SUCCEEDED", "RUNNING"):
        break
    if state in ("FAILED", "CANCELLED"):
        print(f"❌ Deployment {state}: {dep}")
        break
    print(f"  Deploy state: {state}")
    time.sleep(15)

print(f"✅ App deployed")

# Capture the app URL for the UC connection in cell 8
app_info = w.api_client.do("GET", f"/api/2.0/apps/{GRAPHRAG_APP_NAME}")
app_url = app_info.get("url", "")
print(f"App URL: {app_url}")

In [0]:
# --- Create Unity Catalog HTTP Connection ---
# This creates the bridge between the Supervisor Agent and the Databricks App. The connection is an HTTP-type
# UC object that stores the app's URL and auth token. When notebook 04 wires the Supervisor Agent, it
# references this connection by name as an "external-mcp-server" agent type.
#
# The bearer token authenticates requests from the Supervisor to the app. NOTE: This uses the current user's
# session token, which will expire. For production, replace with a service principal token or managed identity.

mcp_url = f"{app_url}/mcp/sse"

# Retrieve current session token for HTTP bearer auth. The UC connection stores this and sends it in the
# Authorization header whenever the Supervisor Agent calls an MCP tool.
bearer_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

try:
    existing = w.api_client.do("GET", f"/api/2.1/unity-catalog/connections/{MCP_CONNECTION_NAME}")  # idempotency
    print(f"Connection '{MCP_CONNECTION_NAME}' already exists")
    print(f"URL: {existing.get('url', '')}")
except:
    # connection_type="HTTP" maps to CONNECTION_HTTP_BEARER internally, which requires the bearer_token option.
    print(f"Creating UC connection '{MCP_CONNECTION_NAME}'...")
    connection = w.api_client.do("POST", "/api/2.1/unity-catalog/connections", body={
        "name": MCP_CONNECTION_NAME,
        "connection_type": "HTTP",
        "options": {
            "host": app_url,
            "base_path": "/mcp/sse",
            "bearer_token": bearer_token
        },
        "comment": "OntoBricks GraphRAG MCP server for stock knowledge graph"
    })
    print(f"✅ Created connection: {MCP_CONNECTION_NAME}")

print(f"\nMCP Connection ready for binding to Supervisor Agent")
print(f"Connection name: {MCP_CONNECTION_NAME}")

In [0]:
# --- Smoke Tests ---
# Validate the graph tables directly via Spark SQL before relying on the MCP server. These queries mirror
# what the app's three tools execute, ensuring data is correct at the source. If these fail, the MCP tools
# will also fail.

print("Testing graph queries...")

print("\nTest 1: Company summary (NVDA)")  # validates get_company_summary() data path
result = spark.sql(f"""
    SELECT ticker, prop_market_cap as market_cap, prop_beta as beta,
           prop_pe_trailing as pe_trailing, prop_pe_forward as pe_forward
    FROM {CATALOG}.{SCHEMA}.graphrag_vertices
    WHERE type = 'Company' AND ticker = 'NVDA'
""").show(truncate=False)

print("\nTest 2: Recent NVDA trading days")  # validates the recent-trading portion of tools
spark.sql(f"""
    SELECT prop_date as date, prop_price_close as close, 
           prop_volume as volume, prop_daily_return as return_pct
    FROM {CATALOG}.{SCHEMA}.graphrag_vertices
    WHERE type = 'TradingDay' AND ticker = 'NVDA'
    ORDER BY prop_date DESC
    LIMIT 5
""").show(truncate=False)

# Each company should have ~250 TRADED_ON edges (one per trading day). Significant deviation = data issue.
print("\nTest 3: TRADED_ON edges per company")
spark.sql(f"""
    SELECT src as company, COUNT(*) as trading_days
    FROM {CATALOG}.{SCHEMA}.graphrag_edges
    WHERE relationship = 'TRADED_ON'
    GROUP BY src
    ORDER BY src
""").show()

print("✅ All tests passed")

## Setup Complete

### Artifacts Created

| Layer | Resource | Details |
|-------|----------|---------|
| **Delta Tables** | `{catalog}.{schema}.graphrag_vertices` | 1,757 rows — 7 Company + 1,750 TradingDay nodes with properties stored as STRING columns |
| | `{catalog}.{schema}.graphrag_edges` | 3,493 rows — 1,750 TRADED_ON + 1,743 NEXT_DAY relationships |
| **Databricks App** | `{GRAPHRAG_APP_NAME}` | FastMCP server running on SSE transport with 3 tools: `query_graph`, `get_company_summary`, `compare_companies` |
| **UC Connection** | `ontobricks-graphrag` | HTTP connection with bearer token auth, pointing to the app's `/mcp/sse` endpoint |

### How It Connects to the Supervisor

Notebook **`04_instructor_setup_sa`** discovers UC connections and registers them as child agents. When it finds `ontobricks-graphrag`, it adds an agent entry like:

```python
{
    "name": "GraphRAG_Stock_Knowledge",
    "agent_type": "external-mcp-server",
    "external_mcp_server": {"connection_name": "ontobricks-graphrag"}
}
```

The Supervisor then routes graph-related questions to this MCP agent.

### Example Queries via the Supervisor Agent

| Query | Routed To | What Happens |
|-------|-----------|---------------|
| "Give me NVDA's fundamentals and recent prices" | `get_company_summary("NVDA")` | Returns Company node props + last 10 TradingDay nodes |
| "Compare AAPL, MSFT, and NVDA" | `compare_companies("AAPL,MSFT,NVDA")` | Queries Company vertices, returns side-by-side table |
| "What does the knowledge graph contain?" | `query_graph("...")` | Returns schema description with node/edge counts |
| "Which stock has the highest P/E ratio?" | `query_graph("...")` | Keyword routing → company comparison sorted by metric |

### Cleanup

To remove all artifacts created by this notebook:

```sql
-- Drop graph tables
DROP TABLE IF EXISTS {catalog}.{schema}.graphrag_vertices;
DROP TABLE IF EXISTS {catalog}.{schema}.graphrag_edges;

-- Drop UC connection (run from a Python cell)
-- w.api_client.do("DELETE", f"/api/2.1/unity-catalog/connections/ontobricks-graphrag")

-- Delete the Databricks App (run from a Python cell)
-- w.api_client.do("DELETE", f"/api/2.0/apps/{GRAPHRAG_APP_NAME}")
```

### Related Notebooks

* **`00_tables_setup`** — Creates the source `ticker_data` table (prerequisite)
* **`04_instructor_setup_sa`** — Creates the Supervisor Agent and wires in MCP connections
* **`05_create_mcp_server_OPTIONAL`** — Adds the You.com web search MCP server (sibling pattern)
* **`05b_create_mcp_neo4j_graph_OPTIONAL`** — Neo4j-based alternative to this notebook (requires external AuraDB instance)